# Wikipedia Prerequisites Extraction using vLLM
## Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4 모델을 사용한 선수 개념 추출

## 1. 라이브러리 설치 및 임포트

In [1]:
import pandas as pd
import json
from vllm import LLM, SamplingParams
from tqdm import tqdm
from typing import List, Dict
import ast

## 2. 프롬프트 템플릿 정의

In [2]:
SYSTEM_PROMPT = """You are an expert curriculum designer for technical topics. Given a Wikipedia article title and its linked topics, identify the most essential prerequisite concepts that must be studied to understand the main topic.

## Task
Analyze the provided title and its associated links, then select up to 20 prerequisite topics that are:
1. **Foundational**: Core concepts needed to understand the main topic
2. **Progressive**: Ordered from basic to advanced when possible
3. **Essential**: Cannot be skipped without leaving knowledge gaps

## Selection Criteria
- Prioritize fundamental concepts over advanced applications
- Include theoretical foundations before practical implementations
- Exclude tangential or overly specific topics
- Focus on direct dependencies, not distant relations
- Limit to maximum 20 items

## Output Format
Return ONLY a valid JSON object with this structure:
{
  "title": "<original title>",
  "prerequisites": [
    {
      "rank": 1,
      "topic": "<topic name>",
      "rationale": "<brief explanation in Korean>"
    }
  ]
}
"""

def create_user_prompt(title: str, links: List[str]) -> str:
    links_str = ", ".join([f'"{link}"' for link in links[:200]])  # 너무 긴 경우 제한
    
    return f"""Analyze the following Wikipedia article:

Title: "{title}"
Links: [{links_str}]

Select up to 20 most essential prerequisite topics from the links list. Return only valid JSON."""

def format_messages(title: str, links: List[str]) -> List[Dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": create_user_prompt(title, links)}
    ]

## 3. 데이터 로드 및 전처리

In [3]:
# CSV 파일 로드
df = pd.read_csv('/data/ephemeral/home/8003_minwoo/PTMT-GraphDB/data/wikipedia/wiki_links.csv')

print(f"Total articles: {len(df)}")
print(f"\nFirst 3 rows:")
print(df.head(3))

Total articles: 5863

First 3 rows:
                                               title  \
0  Lists of open-source artificial intelligence s...   
1                       Love Me (2024 American film)   
2                                  Wiring (software)   

                                               links  
0  ['.NET', '15.ai', '3D optical data storage', '...  
1  ['2024 Sundance Film Festival', '2AM (company)...  
2  ['.NET Gadgeteer', '6LoWPAN', 'ANT+', 'AVR Lib...  


In [4]:
# links 컬럼을 문자열에서 리스트로 변환
def parse_links(links_str: str) -> List[str]:
    """문자열 형태의 리스트를 실제 리스트로 변환"""
    try:
        return ast.literal_eval(links_str)
    except:
        return []

df['links_parsed'] = df['links'].apply(parse_links)

# 링크가 있는 항목만 필터링
df_filtered = df[df['links_parsed'].apply(len) > 0].reset_index(drop=True)

print(f"Articles with links: {len(df_filtered)}")

Articles with links: 5856


## 4. vLLM 모델 로드

In [5]:
# 모델 설정
MODEL_NAME = "JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4"

llm = LLM(
    model=MODEL_NAME,
    trust_remote_code=True,
    max_model_len=4096,
    gpu_memory_utilization=0.9,
    tensor_parallel_size=1,
    enable_prefix_caching=True,
    enforce_eager=True,
)

sampling_params = SamplingParams(
    temperature=0.2,
    top_p=0.9,
    max_tokens=4096,
)

print("Model loaded successfully!")

INFO 01-26 02:24:07 [utils.py:263] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'enable_prefix_caching': True, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 01-26 02:24:13 [model.py:530] Resolved architecture: Qwen3MoeForCausalLM
WARNING 01-26 02:24:13 [model.py:1817] Your device 'Tesla V100-SXM2-32GB' (with compute capability 7.0) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 01-26 02:24:13 [model.py:1869] Casting torch.bfloat16 to torch.float16.
INFO 01-26 02:24:13 [model.py:1545] Using max model len 4096
INFO 01-26 02:24:16 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 01-26 02:24:16 [gptq.py:99] Currently, the 4-bit gptq_gemm kernel for GPTQ is buggy. Please switch to gptq_marlin or gptq_bitblas.


Parse safetensors files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO 01-26 02:24:24 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 01-26 02:24:24 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
WARNING 01-26 02:24:24 [vllm.py:665] Enforce eager set, overriding optimization level to -O0
INFO 01-26 02:24:24 [vllm.py:765] Cudagraph is disabled under eager mode
WARNING 01-26 02:24:26 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=95473) INFO 01-26 02:24:37 [core.py:97] Initializing a V1 LLM engine (v0.14.1) with config: model='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', speculative_config=None, tokenizer='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch

(EngineCore_DP0 pid=95473) /data/ephemeral/home/8003_minwoo/PTMT-GraphDB/.venv/lib/python3.10/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=95473) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=95473)   warnings.warn(


(EngineCore_DP0 pid=95473) INFO 01-26 02:24:41 [cuda.py:351] Using TRITON_ATTN attention backend out of potential backends: ('TRITON_ATTN', 'FLEX_ATTENTION')


Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:03<00:15,  3.85s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:04<00:05,  1.98s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:08<00:06,  3.06s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:13<00:03,  3.59s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:17<00:00,  3.88s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:17<00:00,  3.54s/it]
(EngineCore_DP0 pid=95473) 


(EngineCore_DP0 pid=95473) INFO 01-26 02:25:07 [default_loader.py:291] Loading weights took 17.78 seconds
(EngineCore_DP0 pid=95473) INFO 01-26 02:25:08 [gpu_model_runner.py:3905] Model loading took 15.59 GiB memory and 29.567903 seconds
(EngineCore_DP0 pid=95473) WARNING 01-26 02:25:09 [fused_moe.py:1090] Using default MoE config. Performance might be sub-optimal! Config file not found at /data/ephemeral/home/8003_minwoo/PTMT-GraphDB/.venv/lib/python3.10/site-packages/vllm/model_executor/layers/fused_moe/configs/E=128,N=768,device_name=Tesla_V100-SXM2-32GB,dtype=int4_w4a16.json
(EngineCore_DP0 pid=95473) INFO 01-26 02:25:27 [gpu_worker.py:358] Available KV cache memory: 11.53 GiB
(EngineCore_DP0 pid=95473) INFO 01-26 02:25:27 [kv_cache_utils.py:1305] GPU KV cache size: 125,888 tokens
(EngineCore_DP0 pid=95473) INFO 01-26 02:25:27 [kv_cache_utils.py:1310] Maximum concurrency for 4,096 tokens per request: 30.73x
(EngineCore_DP0 pid=95473) INFO 01-26 02:25:27 [core.py:273] init engine (p

In [6]:
example = df.iloc[4456]
print(example)
messages = format_messages(example['title'], example['links_parsed'])

prompt = llm.get_tokenizer().apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

outputs = llm.generate([prompt], sampling_params)
result = outputs[0].outputs[0].text

title                                 Transformer (deep learning)
links           ['15.ai', 'AAAI Conference on Artificial Intel...
links_parsed    [15.ai, AAAI Conference on Artificial Intellig...
Name: 4456, dtype: object


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [7]:
result

'{\n  "title": "Transformer (deep learning)",\n  "prerequisites": [\n    {\n      "rank": 1,\n      "topic": "Artificial neural network",\n      "rationale": "Transformer 모델은 인공 신경망의 일종으로, 신경망의 기본 구조와 작동 원리 이해가 필수적이다."\n    },\n    {\n      "rank": 2,\n      "topic": "Deep learning",\n      "rationale": "Transformer는 딥러닝의 핵심 기술 중 하나이므로, 딥러닝의 기본 개념과 접근법을 이해해야 한다."\n    },\n    {\n      "rank": 3,\n      "topic": "Attention (machine learning)",\n      "rationale": "Transformer의 핵심 메커니즘인 \'주의\' (attention) 기술을 이해하지 않으면 모델의 작동 원리를 파악할 수 없다."\n    },\n    {\n      "rank": 4,\n      "topic": "Backpropagation",\n      "rationale": "딥러닝 모델의 학습 과정을 가능하게 하는 기초 알고리즘으로, Transformer의 학습 메커니즘 이해에 필수적이다."\n    },\n    {\n      "rank": 5,\n      "topic": "Gradient descent",\n      "rationale": "모델의 가중치 최적화를 위한 기본 최적화 기법으로, Transformer의 학습 과정을 이해하기 위해 필요하다."\n    },\n    {\n      "rank": 6,\n      "topic": "Feedforward neural network",\n      "rationale": "Transformer의 인코더 및 디코더 블록 내부의 기본 구조인 피드포워드 네트워

## 5. 배치 추론 실행

In [8]:
def parse_json_response(text: str) -> Dict:
    """모델 출력에서 JSON 추출 및 파싱"""
    try:
        # JSON 코드 블록 제거
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0]
        elif "```" in text:
            text = text.split("```")[1].split("```")[0]
        
        # JSON 파싱
        return json.loads(text.strip())
    except Exception as e:
        print(f"Parsing error: {e}")
        print(f"Text: {text[:200]}...")
        return None

In [9]:
# 프롬프트 생성
prompts = []
for idx, row in df_filtered.iterrows():
    messages = format_messages(row['title'], row['links_parsed'])
    
    # Qwen 모델의 chat template 적용
    prompt = llm.get_tokenizer().apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    prompts.append(prompt)

print(f"Generated {len(prompts)} prompts")
print(f"\nExample prompt (first 500 chars):\n{prompts[0][:500]}...")

Generated 5856 prompts

Example prompt (first 500 chars):
<|im_start|>system
You are an expert curriculum designer for technical topics. Given a Wikipedia article title and its linked topics, identify the most essential prerequisite concepts that must be studied to understand the main topic.

## Task
Analyze the provided title and its associated links, then select up to 20 prerequisite topics that are:
1. **Foundational**: Core concepts needed to understand the main topic
2. **Progressive**: Ordered from basic to advanced when possible
3. **Essential**...


In [10]:
# 결과 수집
results = []
failed_indices = []

for idx, i in enumerate(range(0, len(prompts), 500)):
    batch_prompts = prompts[i:i+500]
    print(f"Starting batch {idx} inference...")
    outputs = llm.generate(batch_prompts, sampling_params)
    print("Inference completed!")

    for idx, output in enumerate(tqdm(outputs, desc="Parsing results")):
        generated_text = output.outputs[0].text
        parsed_result = parse_json_response(generated_text)
        
        if parsed_result:
            results.append({
                'original_title': df_filtered.iloc[idx]['title'],
                'result': parsed_result,
                'raw_output': generated_text
            })
        else:
            failed_indices.append(idx)
            results.append({
                'original_title': df_filtered.iloc[idx]['title'],
                'result': None,
                'raw_output': generated_text
            })

    print(f"\nSuccessfully parsed: {len(results) - len(failed_indices)}/{len(results)}")
    print(f"Failed to parse: {len(failed_indices)}")

Starting batch 0 inference...


Adding requests:   0%|          | 0/500 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/500 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

ERROR 01-26 03:00:08 [core_client.py:610] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.


KeyboardInterrupt: 

In [ ]:
# 결과를 DataFrame으로 변환
results_df = pd.DataFrame(results)

# JSON 결과 저장
with open('prerequisites_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# CSV 형태로도 저장 (분석용)
flattened_results = []
for result in results:
    if result['result'] and 'prerequisites' in result['result']:
        for prereq in result['result']['prerequisites']:
            flattened_results.append({
                'title': result['original_title'],
                'rank': prereq.get('rank', 0),
                'prerequisite_topic': prereq.get('topic', ''),
                'rationale': prereq.get('rationale', '')
            })

flattened_df = pd.DataFrame(flattened_results)
flattened_df.to_csv('prerequisites_flattened.csv', index=False, encoding='utf-8')

print("Results saved to:")
print("- prerequisites_results.json (full results)")
print("- prerequisites_flattened.csv (flattened for analysis)")

## 6. 결과 분석

In [ ]:
# 통계 확인
print("=== Statistics ===")
print(f"Total articles processed: {len(results)}")
print(f"Successful extractions: {len(results) - len(failed_indices)}")
print(f"Failed extractions: {len(failed_indices)}")

if len(flattened_df) > 0:
    print(f"\nTotal prerequisites extracted: {len(flattened_df)}")
    print(f"Average prerequisites per article: {len(flattened_df) / (len(results) - len(failed_indices)):.2f}")
    print(f"\nTop 10 most common prerequisites:")
    print(flattened_df['prerequisite_topic'].value_counts().head(10))

In [ ]:
# 샘플 결과 확인
print("=== Sample Results ===")
for i in range(min(3, len(results))):
    if results[i]['result']:
        print(f"\n[{i+1}] {results[i]['original_title']}")
        print("Prerequisites:")
        for prereq in results[i]['result'].get('prerequisites', [])[:5]:
            print(f"  {prereq.get('rank', '?')}. {prereq.get('topic', 'N/A')}")
            print(f"     → {prereq.get('rationale', 'N/A')}")

In [ ]:
# 실패한 케이스 확인
if failed_indices:
    print("\n=== Failed Cases ===")
    for idx in failed_indices[:3]:
        print(f"\nTitle: {df_filtered.iloc[idx]['title']}")
        print(f"Raw output: {results[idx]['raw_output'][:300]}...")